# Earnings Call (Opportunity Day) Calendar

Fetch the SET **Earnings Call (OPPDAY)** calendar with `settfex`: symbol, company, date,
video-clip duration, and a ready-to-use YouTube link — as typed models or a pandas DataFrame.

> The opportunity-day backend is stateless (no cookies/bot wall), so these calls are fast and
> need no session warm-up.

## Setup

The DataFrame helpers need pandas, an optional extra:

```bash
pip install "settfex[dataframe]"   # or: uv add pandas
```

In [ ]:
from settfex.services.set import (
    EarningsCallService,
    get_earnings_calls,
    get_earnings_calls_dataframe,
)

## 1. Fetch the current OPPDAY list

`get_earnings_calls()` returns one page of typed `EarningsCallItem`s.

In [ ]:
response = await get_earnings_calls(page_size=12)

print(f"This page: {response.count} items | total available: {response.no_records}")


for item in response.items[:5]:
    print(f"{item.symbol:8} {item.meeting_date.date()}  {item.period}  {item.company_name_clean}")

    print(f"         {item.youtube_url}")

## 2. As a pandas DataFrame

The default columns are exactly `stock_name, company_name, earnings_call_date, video_clip_time, youtube_url`.

In [ ]:
df = await get_earnings_calls_dataframe(page_size=12)

df

## 3. Search a single company (keyword)

`keyword` is a free-text symbol/company search (normalized to uppercase).

In [ ]:
hann = await get_earnings_calls(keyword="HANN")

hann.to_dataframe()

## 4. Filter by quarter and type

Use the filter helpers to discover valid ids. `type_id` 1 = Earnings Call (OPPDAY),
2 = Digital Roadshow, 3 = C-Sign. `quarter_id` 0 = all quarters.

In [ ]:
service = EarningsCallService()


years = await service.fetch_filter_years()

print("Available quarters (first 5):")

for y in years[:5]:
    print(f"  {y.id}  {y.name}")


# Pick the most recent quarter and fetch its OPPDAY presentations

latest_quarter = years[0].id

by_quarter = await service.fetch_earnings_calls(type_id=1, quarter_id=latest_quarter, page_size=20)

print(f"\n{by_quarter.count} item(s) for quarter id {latest_quarter}")

by_quarter.to_dataframe()

## 5. Fetch the whole archive (concurrent) and export to CSV



`get_all_earnings_calls` fetches pages **concurrently** (fast) and can show a `tqdm`

progress bar with `progress=True`. Bound large runs with `max_records` / `max_pages`.

In [ ]:
# get_all_earnings_calls fetches pages concurrently (fast). progress=True shows a
# tqdm bar (pip install "settfex[progress]"). Bound large runs with max_records.
from settfex.services.set import get_all_earnings_calls

full = await get_all_earnings_calls(max_records=200, progress=True)
df_full = full.to_dataframe()
print(f"Collected {len(df_full)} rows")

df_full.to_csv("earnings_calls.csv", index=False)
print("Saved earnings_calls.csv")
df_full.head()

## 6. (Optional) Enrich with detail

`enrich=True` fetches each item's detail concurrently (bounded), adding the YouTube embed
link, the Thai company name, and the meeting **clock-time range** (`detail.meeting_time`,
e.g. `"16:15 - 17:00"`). Note this is *different* from `video_clip_time`, which is the clip
**duration** (`"45:00"`) from the list endpoint.

In [ ]:
enriched = await get_earnings_calls(page_size=5, enrich=True)

for item in enriched.items:
    if item.detail:
        print(f"{item.symbol:8} duration={item.period}  meeting_time={item.detail.meeting_time}")

        print(f"         {item.detail.video_link}")

## 7. Thai transcripts (text for AI)



For each earnings call that has a YouTube video, fetch its **Thai subtitle text** as a plain

string — the raw text you'd feed to an LLM. Needs `pip install "settfex[transcript]"`.



> YouTube rate-limits/IP-blocks aggressively — use this on a **filtered** set (here: one

> company), not the whole archive. A blocked/missing transcript just comes back as `None`.

In [ ]:
from settfex.services.set import fetch_transcripts, get_earnings_calls

scb = await get_earnings_calls(keyword="SCB", page_size=5)

await fetch_transcripts(scb.items, progress=True)


for item in scb.items:
    if item.transcript:
        print(f"{item.symbol} {item.meeting_date.date()} ({len(item.transcript)} chars)")

        print(item.transcript[:300], "...\n")